# Notebook 3: Recursive Self-Calls

In this notebook, you'll learn:
- The key RLM insight: an LLM can **call itself** as a function
- How `llm_query()` works — injected into the sandbox for recursive sub-calls
- The recursion flow: Root LLM → code → `llm_query()` → Sub-LLM → result → sandbox
- How to track and visualize the recursion tree
- Termination: `FINAL()` and `FINAL_VAR()` markers

## The Big Idea

What if the model could say: "I don't know the answer yet, but if I break this into smaller pieces and ask myself about each piece, I can figure it out"?

That's exactly what RLMs do. The model writes code that calls `llm_query()` — which triggers another LLM call. That sub-call can also call `llm_query()`, creating a recursion tree.

## Step 1: The Recursion Flow

Here's how a recursive LLM call works:

```
User Question: "What is the secret code in this document?"
         │
         ▼
    ┌─────────────┐
    │  Root LLM   │  Receives: query + metadata (context length, etc.)
    │             │  Does NOT see the full context directly
    └─────┬───────┘
          │ Generates Python code:
          │   lines = context.split('\n')
          │   for chunk in [lines[:50], lines[50:]]:
          │       result = llm_query(f"Find secret in: {chunk}")
          │
          ▼
    ┌─────────────┐     ┌─────────────┐
    │ Sub-LLM #1  │     │ Sub-LLM #2  │
    │ (chunk 1)   │     │ (chunk 2)   │
    └─────┬───────┘     └─────┬───────┘
          │                   │
          ▼                   ▼
    "No secret found"   "Found: BLUE-FALCON-42"
          │                   │
          └───────┬───────────┘
                  ▼
         Root LLM combines results:
         FINAL("BLUE-FALCON-42")
```

## Step 2: Manual Simulation

Before using the full engine, let's manually simulate what happens. We'll play the role of the orchestrator:

In [ ]:
import sys
sys.path.insert(0, "..")

from rlm_core import LLMClient, Sandbox

# For this demo, we'll simulate LLM responses
# (Replace with real LLM client when vLLM is running)

class SimulatedClient:
    """Simulates LLM responses for demonstration."""
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += 10
        self.total_completion_tokens += 5
        
        class Result:
            prompt_tokens = 10
            completion_tokens = 5
        
        r = Result()
        # Simulate different responses based on the query
        if "secret" in prompt.lower() and "BLUE" in prompt:
            r.text = 'FINAL("BLUE-FALCON-42")'
        elif "secret" in prompt.lower():
            r.text = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
if match:
    FINAL(match.group(1))
else:
    FINAL("No secret found in this section")'''
        else:
            r.text = 'FINAL("I can answer general questions")'
        return r

client = SimulatedClient()
print("Simulated client ready. (Replace with real LLMClient when vLLM is running)")

## Step 3: The llm_query() Function

The `llm_query()` function is what makes recursion possible. It's injected into the sandbox so the model's code can call it. When called, it:

1. Creates a new sub-LLM call
2. Runs it through the same pipeline (sandbox + code execution)
3. Returns the result back to the calling code

In [ ]:
# Let's build llm_query step by step

def simple_llm_query(question):
    """A simplified version of what llm_query does."""
    print(f"  [SUB-CALL] Question: {question}")
    
    # In real RLM: this would call the LLM, get code, execute it
    # For now, we simulate it
    response = client.completion(question)
    print(f"  [SUB-CALL] Response: {response.text[:60]}...")
    
    # Extract answer from FINAL()
    import re
    match = re.search(r'FINAL\("([^"]*)"\)', response.text)
    if match:
        return match.group(1)
    return response.text

# Test it
result = simple_llm_query("What is the secret code?")
print(f"\nReturned: {result}")

## Step 4: Wiring It All Together

Now let's inject `llm_query` into a sandbox alongside `context`, just like the real RLM does:

In [ ]:
# Load our needle-in-haystack document
with open("../data/samples/needle_haystack.txt") as f:
    document = f.read()

print(f"Document length: {len(document)} characters")
print(f"First 200 chars: {document[:200]}...")

# Create a sandbox with both context AND llm_query
sb = Sandbox(variables={
    "context": document,
    "llm_query": simple_llm_query,
})

# Simulate what the model's code might look like
code = '''
# The model examines the context and searches for the secret
import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
if match:
    print(f"Found directly: {match.group(1)}")
else:
    # If direct search fails, try asking a sub-LLM
    result = llm_query("Search the context for any secret code or hidden message")
    print(f"Sub-query found: {result}")
'''

result = sb.execute(code)
print("\n=== Sandbox Output ===")
print(result.stdout)

## Step 5: Tracking the Recursion Tree

Every call creates a node in a tree. We use `RecursionNode` to track:
- What question was asked
- What code was generated
- What the result was
- What sub-calls were made (children)

In [ ]:
from rlm_core import RecursionNode
from rlm_core.visualizer import tree_to_text

# Manually build a sample recursion tree
root = RecursionNode(
    query="What is the secret code in this document?",
    depth=0,
    result="BLUE-FALCON-42"
)

child1 = RecursionNode(
    query="Search first half for secret code",
    depth=1,
    result="No secret found"
)

child2 = RecursionNode(
    query="Search second half for secret code",
    depth=1,
    result="BLUE-FALCON-42"
)

root.children = [child1, child2]

# Visualize it!
print(tree_to_text(root))

## Step 6: FINAL() and FINAL_VAR()

The model signals it's done in two ways:

- **`FINAL("answer")`** — The answer is a literal string
- **`FINAL_VAR(variable_name)`** — The answer is stored in a variable (useful for long outputs)

In [ ]:
# Demonstrate FINAL() and FINAL_VAR()
sb = Sandbox()

# FINAL with a literal
final_result = {"value": None}
def final_fn(answer):
    final_result["value"] = str(answer)
def final_var_fn(var_name):
    final_result["value"] = str(sb.get_variable(var_name))

sb._globals["FINAL"] = final_fn
sb._globals["FINAL_VAR"] = final_var_fn

# Method 1: FINAL() with literal
sb.execute('FINAL("The answer is 42")')
print(f"FINAL() result: {final_result['value']}")

# Reset
final_result["value"] = None

# Method 2: FINAL_VAR() with variable
sb.execute('''
results = []
for i in range(5):
    results.append(f"Item {i}")
answer = ", ".join(results)
FINAL_VAR("answer")
''')
print(f"FINAL_VAR() result: {final_result['value']}")

## Step 7: Depth Limits — Preventing Infinite Recursion

Without limits, the model could recurse forever. The `max_depth` parameter prevents this:

In [ ]:
# Show what happens with depth limits
from rlm_core import RLMEngine

engine = RLMEngine(client=client, max_depth=3)
print(f"Max depth set to: {engine.max_depth}")
print(f"This means: Root (depth 0) → Child (1) → Grandchild (2) → Great-grandchild (3) → STOP")
print(f"\nAny call beyond depth 3 will return an error message instead of recursing further.")

## Step 8: The Full RLMEngine

Let's use the complete `RLMEngine` to run a real recursive query:

In [ ]:
from rlm_core import RLMEngine
from rlm_core.visualizer import tree_to_text

# Run a query through the full engine
engine = RLMEngine(client=client, max_depth=3)
result = engine.run(
    query="What is the secret code hidden in this document?",
    context=document
)

print(f"Answer: {result.answer}")
print(f"Total LLM calls: {result.total_llm_calls}")
print(f"Max depth reached: {result.max_depth_reached}")
print(f"\n=== Recursion Tree ===")
print(tree_to_text(result.root_node))

## Step 9: Visualizing with Graphviz

For more complex recursion trees, we can generate visual diagrams:

In [ ]:
from rlm_core.visualizer import tree_to_graphviz, tree_to_dict
import json

# Generate graphviz DOT format
dot_string = tree_to_graphviz(result.root_node)
print("=== Graphviz DOT ===")
print(dot_string)

print("\n=== JSON Tree ===")
print(json.dumps(tree_to_dict(result.root_node), indent=2))

## Exercise

Try running the RLM engine on the aggregation dataset. Load `aggregation_items.json` and ask "How many items are red and large?"

In [ ]:
import json

# Load the aggregation dataset
with open("../data/samples/aggregation_items.json") as f:
    items = json.load(f)

context = json.dumps(items, indent=2)
print(f"Loaded {len(items)} items ({len(context)} chars)")

# Run through the engine
result = engine.run(
    query="How many items in the dataset are both red AND large?",
    context=context
)

print(f"\nAnswer: {result.answer}")
print(f"LLM calls: {result.total_llm_calls}")
print(f"\n=== Recursion Tree ===")
print(tree_to_text(result.root_node))

# Verify against ground truth
actual = len([i for i in items if i["color"] == "red" and i["size"] == "large"])
print(f"\nGround truth: {actual} items are red AND large")

## Key Takeaways

1. **`llm_query()` enables recursion** — it's a function injected into the sandbox that calls back to the LLM
2. **The recursion tree** tracks every call — who asked what, what code ran, what the result was
3. **`FINAL()` and `FINAL_VAR()`** signal the model is done — they're how answers propagate up the tree
4. **Depth limits** prevent infinite recursion — essential safety mechanism
5. **The RLMEngine** orchestrates everything — sandbox, LLM calls, recursion tracking, and error handling

## What's Next?

In Notebook 4, we'll explore the **emergent strategies** — the clever patterns (peeking, grepping, partition+map) that models naturally discover when given this recursive power.